In [60]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import cv2
import os
from skimage.feature import local_binary_pattern

In [61]:
def gabor_features(img):
    gabor_filters = []
    ksize = 31
    for theta in np.arange(0, np.pi, np.pi / 8):
        for sigma in (1, 3):
            for lamda in np.arange(np.pi/4, np.pi, np.pi/4):
                for gamma in (0.5, 0.8):
                    gabor_filters.append(cv2.getGaborKernel((ksize, ksize), sigma, theta, lamda, gamma, 0))

    features = []
    for kernel in gabor_filters:
        filtered = cv2.filter2D(img, cv2.CV_8UC3, kernel)
        features.append(filtered.mean())
    
    return np.array(features)

In [62]:
def lbp_features(img):
    radius = 1
    n_points = 8 * radius
    lbp = local_binary_pattern(img, n_points, radius, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    return hist / hist.sum()

In [63]:
def extract_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gabor = gabor_features(gray)
    lbp = lbp_features(gray)
    return np.hstack((gabor, lbp))

In [64]:
class FaceDataset(Dataset):
    def __init__(self, directory):
        
        self.features = []
        self.labels = []
        for label in ['Real', 'Fake']:
            count=1
            folder = os.path.join(directory, label)
            for filename in os.listdir(folder):
                img_path = os.path.join(folder, filename)
                img = cv2.imread(img_path)
                if count % 4000 == 0:
                    break
                if img is not None:
                    count += 1
                    self.features.append(extract_features(img))
                    self.labels.append(1 if label == 'Real' else 0)
                    print(f'File count: {count}')
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return torch.tensor(self.features[index], dtype=torch.float32), torch.tensor(self.labels[index], dtype=torch.long)
                    

In [65]:
directory = r'C:\Users\Ruchir\OneDrive\Desktop\DOG'
dataset = FaceDataset(directory)

File count: 2
File count: 3
File count: 4
File count: 5
File count: 6
File count: 7
File count: 8
File count: 9
File count: 10
File count: 11
File count: 12
File count: 13
File count: 14
File count: 15
File count: 16
File count: 17
File count: 18
File count: 19
File count: 20
File count: 21
File count: 22
File count: 23
File count: 24
File count: 25
File count: 26
File count: 27
File count: 28
File count: 29
File count: 30
File count: 31
File count: 32
File count: 33
File count: 34
File count: 35
File count: 36
File count: 37
File count: 38
File count: 39
File count: 40
File count: 41
File count: 42
File count: 43
File count: 44
File count: 45
File count: 46
File count: 47
File count: 48
File count: 49
File count: 50
File count: 51
File count: 52
File count: 53
File count: 54
File count: 55
File count: 56
File count: 57
File count: 58
File count: 59
File count: 60
File count: 61
File count: 62
File count: 63
File count: 64
File count: 65
File count: 66
File count: 67
File count: 68
Fil

In [89]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

In [90]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [97]:
class NeuralNet(nn.Module):
    
    def __init__(self, input_dim):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 32)
        self.fc4 = nn.Linear(32, 2)
        
    def forward(self, x):
        return self.fc4(F.relu(self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))))
        

In [98]:
model = NeuralNet(106)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [100]:
epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch: {epoch}, Loss: {total_loss/len(train_loader)}')
    
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        predicted = torch.argmax(outputs, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test accuracy: {100 * correct/total}')

Epoch: 0, Loss: 0.10563544049393386
Epoch: 1, Loss: 0.1009305910184048
Epoch: 2, Loss: 0.1028944362537004
Epoch: 3, Loss: 0.10427039614878594
Epoch: 4, Loss: 0.10436185256112367
Epoch: 5, Loss: 0.10675803542602808
Epoch: 6, Loss: 0.10266012696083635
Epoch: 7, Loss: 0.10651940773648676
Epoch: 8, Loss: 0.10265744955744595
Epoch: 9, Loss: 0.10217855678871274
Epoch: 10, Loss: 0.0905806920537725
Epoch: 11, Loss: 0.09681705804541707
Epoch: 12, Loss: 0.10298171386588365
Epoch: 13, Loss: 0.0925422812718898
Epoch: 14, Loss: 0.0967073140712455
Epoch: 15, Loss: 0.09406000539893285
Epoch: 16, Loss: 0.09692333733662963
Epoch: 17, Loss: 0.09329743365524337
Epoch: 18, Loss: 0.09135942656314
Epoch: 19, Loss: 0.09602425369783305
Test accuracy: 95.0625
